In [12]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import HistGradientBoostingRegressor
import librosa
import parselmouth
import maad
from sklearn.model_selection import cross_val_score
from maad.features import temporal_entropy
from scipy.stats import skew, kurtosis

from torchaudio.transforms import Spectrogram, AmplitudeToDB
from torchaudio import load
from torch import Tensor

In [2]:
csvTotale = pd.read_csv('.././data/development.csv', header=0, index_col=0).drop(columns=['sampling_rate'])
csvTotale['gender'] = csvTotale['gender'].map(lambda x: 0 if x=='male' else 1)
csvTotale['tempo'] = csvTotale['tempo'].map(lambda x: float(x[1:-1]))
csvTotale['path'] = csvTotale['path'].map(lambda x: x.split('/')[-1])
csvEval = pd.read_csv('.././data/evaluation.csv', header=0, index_col=0).drop(columns=['sampling_rate'])
csvEval['gender'] = csvEval['gender'].map(lambda x: 0 if x=='male' else 1)
csvEval['tempo'] = csvEval['tempo'].map(lambda x: float(x[1:-1]))
csvEval['path'] = csvEval['path'].map(lambda x: x.split('/')[-1])

In [7]:
def loadDataAsSpectrogram(path:str, csv:pd.DataFrame)->dict[str:Tensor]:
    return {file:AmplitudeToDB()(Spectrogram()(load(path+file)[0]))[0] for file in csv['path']}

In [ ]:
audioDevelopment = loadDataAsSpectrogram('.././data/audios_development/', csvTotale)

In [10]:
audioEval = loadDataAsSpectrogram('.././data/audios_evaluation/', csvEval)

In [125]:
def createSubMatrix(data:Tensor, rows:int, cols:int)->dict[str:np]:
    return ({key:np.array([data[key][data[key].shape[0]//rows * i: min(data[key].shape[0]//rows * (i+1), data[key].shape[0]), data[key].shape[1]//cols * j: min(data[key].shape[1]//cols * (j+1), data[key].shape[1])].mean() for i in range(rows) for j in range(cols)]) for key in data},
            {key:np.array([data[key][data[key].shape[0]//rows * i: min(data[key].shape[0]//rows * (i+1), data[key].shape[0]), data[key].shape[1]//cols * j: min(data[key].shape[1]//cols * (j+1), data[key].shape[1])].std() for i in range(rows) for j in range(cols)]) for key in  data})

In [126]:
audiosDevelopment = createSubMatrix(audioDevelopment, 25, 4)

In [127]:
audiosEval = createSubMatrix(audioEval, 25, 4)

In [128]:
def linearize(data:dict[str:np])->np:    
    return pd.DataFrame({key:np.concatenate((data[0][key].flatten(),
            data[1][key].flatten()), axis=0) for key in data[0]}, 
                        index=[f'mean{i}' for i in range(25*4)]+[f'std{i}' for i in range(25*4)]).T

In [129]:
audiosEval = linearize(audiosEval)


In [130]:
audiosDevelopment = linearize(audiosDevelopment)

In [131]:
audiosEval = pd.concat([audiosEval, csvEval.set_index(csvEval['path'])[['gender', 'hnr', 'shimmer', 'jitter']]], axis=1)
audiosDevelopment = pd.concat([audiosDevelopment, csvTotale.set_index(csvTotale['path'])[['gender', 'hnr', 'shimmer', 'jitter']]], axis=1)

In [132]:
def computeEntropy(paths, addPath):
    def singleEntropy(audio):
        return pd.Series(temporal_entropy(audio), index=['entropy'])
    return pd.DataFrame({file:singleEntropy(librosa.load(addPath+file, sr=22050)[0]) for file in paths}).T

In [133]:
audiosEval = pd.concat([audiosEval, computeEntropy(csvEval['path'], '.././data/audios_evaluation/')], axis=1)
audiosDevelopment = pd.concat([audiosDevelopment, computeEntropy(csvTotale['path'], '.././data/audios_development/')], axis=1)

In [134]:
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor


(cross_val_score(HistGradientBoostingRegressor(), audiosDevelopment, csvTotale['tempo'], cv=20, scoring='neg_mean_absolute_error').mean(),
cross_val_score(SVR(), audiosDevelopment, csvTotale['tempo'], cv=20, scoring='neg_mean_absolute_error').mean(),
cross_val_score(KNeighborsRegressor(n_jobs=-1), audiosDevelopment, csvTotale['tempo'], cv=20, scoring='neg_mean_absolute_error').mean())

(np.float64(-25.70098635715264),
 np.float64(-24.574207854585943),
 np.float64(-26.87895485442411))

In [135]:
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import KFold
from sklearn.pipeline import make_pipeline


(cross_val_score(make_pipeline(StandardScaler(), HistGradientBoostingRegressor()), audiosDevelopment.iloc[:, :-5], csvTotale['tempo'], cv=KFold(n_splits=20, shuffle=True), scoring='neg_mean_absolute_error').mean(),
cross_val_score(make_pipeline(StandardScaler(), SVR()), audiosDevelopment.iloc[:, :-5], csvTotale['tempo'], cv=KFold(n_splits=20, shuffle=True), scoring='neg_mean_absolute_error').mean(),
cross_val_score(make_pipeline(StandardScaler(), KNeighborsRegressor(n_jobs=-1)), audiosDevelopment.iloc[:, :-5], csvTotale['tempo'], cv=KFold(n_splits=20, shuffle=True), scoring='neg_mean_absolute_error').mean())

(np.float64(-25.804070945185845),
 np.float64(-24.574880407999366),
 np.float64(-26.729800828041625))